In [0]:
dbutils.secrets.listScopes()

In [0]:
retailscope

In [0]:
dbutils.secrets.list("retailscope")

In [0]:
dbutils.secrets.list("eventhub-scope1")

In [0]:
CONNECTION_STR = dbutils.secrets.get(
    scope="eventhub-scope1",
    key="storagekey"
)

In [0]:
client_id = dbutils.secrets.get(
    scope="retailscope",
    key="clientid"
)

In [0]:
tenant_id = dbutils.secrets.get(
    scope="retailscope",
    key="tenantid"
)

In [0]:
secret_value = dbutils.secrets.get(
    scope="retailscope",
    key="secretvalue"
)

In [0]:
spark.conf.set(
  "fs.azure.account.key.project1demo.dfs.core.windows.net",
  CONNECTION_STR
)

In [0]:
from pyspark.sql.functions import *

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")

    # FILE NOTIFICATION
    .option("cloudFiles.useNotifications", "true")
    .option("cloudFiles.subscriptionId", "4967f902-d637-449d-81c4-901cd225af2f")
    .option("cloudFiles.tenantId", tenant_id)
    .option("cloudFiles.clientId", client_id)
    .option("cloudFiles.clientSecret", secret_value)
    .option("cloudFiles.resourceGroup", "Realtimedata")
    .option("cloudFiles.queueName", "autoloader-account-queue")

    .option("cloudFiles.schemaLocation", "/Volumes/bankingdata/bronze/schemalocation/account/")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("badRecordsPath", "/Volumes/bankingdata/bronze/badrecords/account/")
    .option("pathGlobFilter", "*.csv")

    .load("abfss://project2@project1demo.dfs.core.windows.net/account/")
)

df = df.withColumn("ingestion_time", current_timestamp()) \
       .withColumn("source_file", input_file_name())

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/bankingdata/bronze/checkpoints/account/") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("bankingdata.bronze.account")

In [0]:
%sql
select * from bankingdata.bronze.account